In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [12]:
import cv2
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [13]:
class FundusDataset(Dataset):
    def __init__(self, img_dir, mask_dir, csv_file):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.data = pd.read_csv(csv_file)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['image_name']

        # IMAGE
        img = cv2.imread(os.path.join(self.img_dir, img_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224,224)).astype(np.float32) / 255.0

        # MASK
        mask = cv2.imread(os.path.join(self.mask_dir, img_name), 0)
        mask = cv2.resize(mask, (224,224)).astype(np.float32) / 255.0

        # CLINICAL
        features = np.array([
            row['thickness'],
            row['age'],
            row['gender']
        ], dtype=np.float32)

        label = int(row['label'])

        return {
            "image": torch.from_numpy(img).permute(2,0,1),
            "mask": torch.from_numpy(mask).unsqueeze(0),
            "features": torch.from_numpy(features),
            "label": torch.tensor(label)
        }

In [14]:
IMG_PATH  = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
MASK_PATH = "/kaggle/input/datasets/amimulahasanrofik/fundusimagevesselenhance"
CSV_PATH  = "/kaggle/input/datasets/amimulahasanrofik/abu-csv/data_info.csv"

dataset = FundusDataset(IMG_PATH, MASK_PATH, CSV_PATH)
loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)

In [15]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 32, 3, padding=1)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.interpolate(x, scale_factor=1)  # safe
        x = F.relu(self.conv3(x))
        return torch.sigmoid(self.out(x))

In [16]:
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()

        # SAFE ResNet (no weight arg)
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Linear(512, 64)

        self.fc1 = nn.Linear(64 + 4, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, img, clinical, vessel_density):
        x = self.backbone(img)

        combined = torch.cat(
            [x, clinical, vessel_density.unsqueeze(1)],
            dim=1
        )

        x = torch.relu(self.fc1(combined))
        return self.fc2(x)

In [17]:
seg_model = UNet().to(device)
clf_model = HybridModel().to(device)

optimizer = torch.optim.Adam(
    list(seg_model.parameters()) + list(clf_model.parameters()),
    lr=1e-4
)

loss_seg = nn.BCELoss()
loss_cls = nn.CrossEntropyLoss()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [18]:
EPOCHS = 5

for epoch in range(EPOCHS):
    seg_model.train()
    clf_model.train()

    total_loss = 0

    for batch in tqdm(loader):
        img = batch["image"].to(device).float()
        mask = batch["mask"].to(device).float()
        clinical = batch["features"].to(device).float()
        label = batch["label"].to(device)

        # Phase 1
        pred_mask = seg_model(img)
        l1 = loss_seg(pred_mask, mask)

        # vessel density
        vd = (pred_mask > 0.5).float().mean(dim=[1,2,3])

        # Phase 2
        output = clf_model(img, clinical, vd)
        l2 = loss_cls(output, label)

        loss = l1 + l2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

  0%|          | 0/1452 [00:00<?, ?it/s]


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
